# Price Prediction Dataset EDA

This notebook visualizes correlations between the actual boundary variables and the standardized price target.

In [4]:
from pathlib import Path
import os


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        dataset_path = candidate / "output/clean_datasets/price_prediction_dataset.csv"
        if dataset_path.exists():
            return candidate
    raise FileNotFoundError("Could not find output/clean_datasets/price_prediction_dataset.csv")


PROJECT_ROOT = find_project_root()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

INPUT_PATH = PROJECT_ROOT / "output/clean_datasets/price_prediction_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "output/EDA_figures/price_prediction"
METHOD = "pearson"

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["font.size"] = 10

print(f"Project root: {PROJECT_ROOT}")
print(f"Input dataset: {INPUT_PATH}")

Project root: /Users/lenovo/Desktop/GRIPS2026_Project
Input dataset: /Users/lenovo/Desktop/GRIPS2026_Project/output/clean_datasets/price_prediction_dataset.csv


## Load Dataset

In [5]:

df = pd.read_csv(INPUT_PATH, parse_dates=["time"])
numeric = df.select_dtypes(include="number")

print(f"Rows: {len(df):,}")
print(f"Numeric columns: {len(numeric.columns)}")
display(df.head())

Rows: 34,944
Numeric columns: 8


,time,System_Load,Wind_And_Solar,Tie_Line,Wind_Power,Photovoltaic,Hydropower,Non-Marketized_Unit,price
0,2025-01-02 00:00:00,2.972606,0.351390,0.051881,0.351626,-0.000236,0.026298,0.440875,1.272545
1,2025-01-02 00:15:00,2.975888,0.360655,0.039225,0.361107,-0.000452,0.026249,0.440386,1.272545
2,2025-01-02 00:30:00,2.969591,0.378003,0.041159,0.378530,-0.000527,0.026497,0.439392,1.272545
3,2025-01-02 00:45:00,2.976092,0.388225,0.028153,0.388598,-0.000373,0.026342,0.435306,1.272545
4,2025-01-02 01:00:00,2.959156,0.396585,0.042151,0.396982,-0.000396,0.026375,0.435755,1.272545


## Correlation Heatmap

In [6]:
corr = numeric.corr(method=METHOD)


def plot_correlation_heatmap(corr: pd.DataFrame, method: str = "pearson"):
    labels = corr.columns.tolist()
    fig, ax = plt.subplots(figsize=(10, 8))
    norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
    image = ax.imshow(corr, cmap="coolwarm", norm=norm)

    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_title(f"{method.title()} Correlation Heatmap")

    for row_idx, row_label in enumerate(labels):
        for col_idx, col_label in enumerate(labels):
            value = corr.loc[row_label, col_label]
            text_color = "white" if abs(value) > 0.65 else "black"
            ax.text(
                col_idx,
                row_idx,
                f"{value:.2f}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=9,
            )

    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Correlation")
    fig.tight_layout()
    return fig, ax


heatmap_fig, heatmap_ax = plot_correlation_heatmap(corr, METHOD)
display(heatmap_fig)

<Figure size 1300x1040 with 2 Axes>

## Correlation With Price

In [7]:
price_corr = corr["price"].drop("price").sort_values(key=lambda values: values.abs(), ascending=True)
colors = ["#b2182b" if value >= 0 else "#2166ac" for value in price_corr]

bar_fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(price_corr.index, price_corr.values, color=colors)
ax.axvline(0, color="#333333", linewidth=1)
ax.set_xlim(-1, 1)
ax.set_xlabel("Correlation with price")
ax.set_title(f"{METHOD.title()} Correlation With Price")

for y_pos, value in enumerate(price_corr.values):
    label_x = value + 0.03 if value >= 0 else value - 0.03
    ha = "left" if value >= 0 else "right"
    ax.text(label_x, y_pos, f"{value:.2f}", va="center", ha=ha)

bar_fig.tight_layout()
display(bar_fig)

<Figure size 1040x624 with 1 Axes>

## Save Figures

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

matrix_path = OUTPUT_DIR / "EDA_price_prediction_correlation_matrix.csv"
heatmap_path = OUTPUT_DIR / "EDA_price_prediction_heatmap.png"
price_corr_path = OUTPUT_DIR / "EDA_price_prediction_price_correlations.png"

corr.to_csv(matrix_path)
heatmap_fig.savefig(heatmap_path, bbox_inches="tight")
bar_fig.savefig(price_corr_path, bbox_inches="tight")

print(f"Saved correlation matrix: {matrix_path}")
print(f"Saved heatmap: {heatmap_path}")
print(f"Saved price-correlation chart: {price_corr_path}")

Saved correlation matrix: /Users/lenovo/Desktop/GRIPS2026_Project/output/EDA_figures/price_prediction/EDA_price_prediction_correlation_matrix.csv
Saved heatmap: /Users/lenovo/Desktop/GRIPS2026_Project/output/EDA_figures/price_prediction/EDA_price_prediction_heatmap.png
Saved price-correlation chart: /Users/lenovo/Desktop/GRIPS2026_Project/output/EDA_figures/price_prediction/EDA_price_prediction_price_correlations.png


## Daily Price Trend

This figure focuses directly on the `price` column from `price_prediction_dataset.csv` and shows the daily price trend. The line is the daily median, and the light band shows the daily interquartile range (25th–75th percentile).

In [ ]:
price_df = df[["time", "price"]].copy()
price_df = price_df.sort_values("time").dropna(subset=["time", "price"])

price_daily = (
    price_df.set_index("time")["price"]
    .resample("D")
    .agg(
        q25=lambda values: values.quantile(0.25),
        median="median",
        q75=lambda values: values.quantile(0.75),
    )
    .dropna(how="all")
)

display(price_daily.describe())

price_fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(
    price_daily.index,
    price_daily["median"],
    color="#1f77b4",
    linewidth=1.8,
    label="Daily median",
)
ax.fill_between(
    price_daily.index,
    price_daily["q25"].to_numpy(),
    price_daily["q75"].to_numpy(),
    color="#1f77b4",
    alpha=0.18,
    linewidth=0,
    label="Daily IQR (25th–75th percentile)",
)

ax.set_title("Daily Price Trend from price_prediction_dataset.csv")
ax.set_xlabel("Time")
ax.set_ylabel("Price")
ax.grid(True, alpha=0.25)
ax.legend()
price_fig.autofmt_xdate()
price_fig.tight_layout()

price_daily_trend_path = OUTPUT_DIR / "EDA_price_prediction_daily_price_trend.png"
price_fig.savefig(price_daily_trend_path, bbox_inches="tight")
display(price_fig)

print(f"Saved daily price trend figure: {price_daily_trend_path}")


## Daily Price Trend: Mean and IQR

This figure shows the daily mean price as the main line, with the daily interquartile range (25th–75th percentile) as the shaded band. This keeps the average daily level visible while still showing the middle spread of 15-minute prices within each day.

In [ ]:
price_daily_mean_iqr = (
    price_df.set_index("time")["price"]
    .resample("D")
    .agg(
        q25=lambda values: values.quantile(0.25),
        mean="mean",
        q75=lambda values: values.quantile(0.75),
    )
    .dropna(how="all")
)

display(price_daily_mean_iqr.describe())

price_mean_iqr_fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(
    price_daily_mean_iqr.index,
    price_daily_mean_iqr["mean"],
    color="#d62728",
    linewidth=1.8,
    label="Daily mean",
)
ax.fill_between(
    price_daily_mean_iqr.index,
    price_daily_mean_iqr["q25"].to_numpy(),
    price_daily_mean_iqr["q75"].to_numpy(),
    color="#d62728",
    alpha=0.16,
    linewidth=0,
    label="Daily IQR (25th–75th percentile)",
)

ax.set_title("Daily Price Trend: Mean and IQR from price_prediction_dataset.csv")
ax.set_xlabel("Time")
ax.set_ylabel("Price")
ax.grid(True, alpha=0.25)
ax.legend()
price_mean_iqr_fig.autofmt_xdate()
price_mean_iqr_fig.tight_layout()

price_daily_mean_iqr_path = OUTPUT_DIR / "EDA_price_prediction_daily_price_mean_iqr.png"
price_mean_iqr_fig.savefig(price_daily_mean_iqr_path, bbox_inches="tight")
display(price_mean_iqr_fig)

print(f"Saved daily price mean/IQR figure: {price_daily_mean_iqr_path}")
